# Data Extraction: Customer Propensity Dataset

This notebook extracts customer features from Google BigQuery's [TheLook E-Commerce](https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce) public dataset for building a repeat purchase propensity model.

**Pipeline:**
1. Connect to BigQuery with cost controls
2. Execute parameterized feature engineering SQL
3. Validate data quality
4. Export to CSV for modeling in `02_propensity_modeling.ipynb`

In [1]:
import pandas as pd
import os
from google.cloud import bigquery
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Config
PROJECT_ID = "churn-portfolio-project"
SQL_FILE_PATH = "../sql/feature_engineering.sql"
DATA_PATH = "../data/raw/propensity_data.csv"

# Parameters
CUTOFF_DATE = "2025-10-01"    # Point-in-time split (simulated "today")
PREDICTION_WINDOW = 90        # Days to define retention

## Target Definition

A customer is labeled **retained** (`is_retained = 1`) if they make a purchase within 90 days of the cutoff date.

| Design Choice | Rationale |
|---------------|----------|
| 90-day window | Industry standard for e-commerce; captures natural purchase cycles while enabling timely intervention |
| Point-in-time features | All features computed *before* cutoff to prevent data leakage |
| 365-day active filter | Excludes users inactive >1 year (already lost, not predictable) |

**Expected retention rate:** ~12%, typical for non-subscription fashion e-commerce where most customers are one-time buyers.

## Feature Engineering

Features are extracted from three source tables:

### Transactional Features (RFM)
*Source: `order_items`*

| Feature | Description | Hypothesis |
|---------|-------------|------------|
| `days_since_last_order` | Recency of last purchase | Lower = more engaged |
| `total_orders` | Purchase frequency | Higher = more loyal |
| `total_spend` | Lifetime monetary value | Weak loyalty signal |
| `avg_order_value` | Spend per transaction | Category preference |
| `returned_orders` | Count of returns | Dissatisfaction signal |
| `return_rate` | Returns as % of orders | Fit/quality issues |

### Demographic Features
*Source: `users`*

| Feature | Description | Hypothesis |
|---------|-------------|------------|
| `customer_tenure_days` | Days since registration | Established vs. new |
| `age` | Customer age | Cohort behavior differences |
| `gender` | M/F | Category preferences |
| `traffic_source` | Acquisition channel | Organic more loyal than paid |
| `country` | Geographic market | Regional retention patterns |

### Engagement Features
*Source: `events`*

| Feature | Description | Hypothesis |
|---------|-------------|------------|
| `total_sessions` | Site visit count | Engagement depth |
| `total_events` | Total interactions | Activity level |
| `days_since_last_visit` | Browsing recency | More sensitive than order recency |
| `avg_events_per_session` | Session depth | Quality of engagement |
| `distinct_event_types` | Behavioral diversity | Exploration breadth |
| `cart_events` | Add-to-cart count | Purchase intent |
| `product_view_events` | Product page views | Browse behavior |

In [3]:
# Load and display SQL query
with open(SQL_FILE_PATH) as f:
    query = f.read()

# Show first 50 lines for reference
print("\n".join(query.split("\n")[:50]))
print("\n... [truncated] ...")

/* 
  Feature Engineering: Customer Retention Propensity Model

  This query builds a customer-level feature set for predicting repeat purchase
  probability within a 90-days window. Features are computed strictly before the
  cutoff date to prevent data leakage.

  Data Sources:
    - order_items: Transaction history (RFM features)
    - users: Demographics and acquisition channel
    - events: Behavioral engagement signals
    - orders: Target variable (future purchases)

  Parameters:
    @cutoff_date: Point-in_time for train/test split (simulated "today")
    @predication_window_days: Days after cutoff to define retention (default: 90)
 */


-- =============================================================================
-- CTE 1: TRANSACTIONAL FEATURES (RFM + Returns)
-- Source: order_items table
-- =============================================================================
WITH user_behavior AS (
  SELECT
    user_id,
    
    -- Recency: Days since last order (lower = more eng

## Query Execution

In [4]:
# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Set up the query job configuration with parameters
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=500 * 1024**2, # 500MB safety cap
    query_parameters=[
        bigquery.ScalarQueryParameter("cutoff_date", "DATE", CUTOFF_DATE),
        bigquery.ScalarQueryParameter("prediction_window_days", "INT64", PREDICTION_WINDOW),
    ],
)

In [5]:
# Dry run: Estimate cost before execution
job_config.dry_run = True
job = client.query(query, job_config=job_config)

bytes_processed = job.total_bytes_processed
mb = bytes_processed / 1e6
cost = bytes_processed / 1e12 * 5  # $5 per TB

print(f"Estimated data scan: {mb:.2f} MB")
print(f"Estimated cost: ${cost:.4f}")

Estimated data scan: 158.06 MB
Estimated cost: $0.0008


In [6]:
# Execute query (or load cached data)
if os.path.exists(DATA_PATH):
    print(f"Loading cached data from {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    if mb > 500:
        raise ValueError(f"Query too large ({mb:.0f} MB). Adjust parameters.")
    
    print("Executing BigQuery...")
    job_config.dry_run = False
    df = client.query(query, job_config=job_config).to_dataframe()
    
    # Cache locally
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Saved to {DATA_PATH}")

print(f"\nDataset shape: {df.shape[0]:,} customers × {df.shape[1]} features")

Executing BigQuery...
Saved to ../data/raw/propensity_data.csv

Dataset shape: 30,549 customers × 21 features


## Data Validation

In [7]:
# Schema check
print("=== Schema ===")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() > 0 else "No missing values")

=== Schema ===
user_id                     Int64
days_since_last_order       Int64
total_orders                Int64
total_spend               float64
avg_order_value           float64
returned_orders             Int64
return_rate               float64
customer_tenure_days        Int64
age                         Int64
gender                     object
traffic_source             object
country                    object
total_sessions              Int64
total_events                Int64
days_since_last_visit       Int64
avg_events_per_session    float64
distinct_event_types        Int64
cart_events                 Int64
product_view_events         Int64
purchase_events             Int64
is_retained                 Int64
dtype: object

=== Missing Values ===
No missing values


In [8]:
# Data quality assertions
assert df['user_id'].is_unique, "Duplicate user IDs found"
assert df['days_since_last_order'].max() < 365, "Active filter not applied correctly"
assert df['is_retained'].isin([0, 1]).all(), "Target variable has invalid values"
assert df['return_rate'].between(0, 1).all(), "Return rate out of bounds"

print("✓ All data quality checks passed")

✓ All data quality checks passed


In [9]:
# Target distribution
retention_rate = df['is_retained'].mean()
print(f"=== Target Distribution ===")
print(f"Retained (1): {df['is_retained'].sum():,} ({retention_rate:.1%})")
print(f"Churned (0):  {(~df['is_retained'].astype(bool)).sum():,} ({1-retention_rate:.1%})")

=== Target Distribution ===
Retained (1): 3,546 (11.6%)
Churned (0):  27,003 (88.4%)


In [10]:
# Numerical feature summary
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
numerical_cols = [c for c in numerical_cols if c not in ['user_id', 'is_retained']]

df[numerical_cols].describe().round(2)

,days_since_last_order,total_orders,total_spend,avg_order_value,returned_orders,return_rate,customer_tenure_days,age,total_sessions,total_events,days_since_last_visit,avg_events_per_session,distinct_event_types,cart_events,product_view_events,purchase_events
count,30549.0,30549.0,30549.00,30549.00,30549.0,30549.00,30549.0,30549.0,30549.0,30549.0,30549.0,30549.00,30549.0,30549.0,30549.0,30549.0
mean,154.36,2.37,141.26,59.54,0.23,0.10,968.37,41.04,2.38,17.07,154.31,6.25,4.8,4.52,4.52,2.37
std,104.3,1.65,141.25,51.80,0.66,0.26,664.55,17.04,1.65,17.21,104.32,2.01,0.4,5.25,5.25,1.65
min,1.0,1.0,0.02,0.02,0.0,0.00,4.0,12.0,1.0,5.0,1.0,5.00,4.0,1.0,1.0,1.0
25%,62.0,1.0,44.95,30.00,0.0,0.00,406.0,26.0,1.0,5.0,62.0,5.00,5.0,1.0,1.0,1.0
50%,140.0,2.0,96.46,46.99,0.0,0.00,815.0,41.0,2.0,10.0,140.0,5.00,5.0,2.0,2.0,2.0
75%,240.0,3.0,191.78,71.50,0.0,0.00,1463.0,56.0,3.0,19.0,240.0,7.00,5.0,5.0,5.0,3.0
max,364.0,13.0,1655.29,999.00,8.0,1.00,2464.0,70.0,13.0,161.0,364.0,13.00,5.0,49.0,49.0,13.0


In [11]:
# Categorical feature summary
categorical_cols = ['gender', 'traffic_source', 'country']

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts().head(10))


=== gender ===
gender
F    15281
M    15268
Name: count, dtype: int64

=== traffic_source ===
traffic_source
Search      21436
Organic      4545
Facebook     1856
Email        1535
Display      1177
Name: count, dtype: int64

=== country ===
country
China             10368
United States      6853
Brasil             4399
South Korea        1613
France             1486
United Kingdom     1452
Spain              1249
Germany            1213
Japan               754
Australia           694
Name: count, dtype: int64


In [12]:
# Sample records
df.head()

,user_id,days_since_last_order,total_orders,total_spend,avg_order_value,returned_orders,return_rate,customer_tenure_days,age,gender,...,country,total_sessions,total_events,days_since_last_visit,avg_events_per_session,distinct_event_types,cart_events,product_view_events,purchase_events,is_retained
0,58455,322,9,538.400000,59.822222,0,0.000000,843,31,F,...,China,9,85,322,9.444444,5,25,25,9,0
1,717,326,6,187.470001,31.245000,0,0.000000,2315,24,F,...,Brasil,6,49,326,8.166667,5,14,14,6,1
2,4067,232,7,343.249998,49.035714,2,0.285714,985,65,F,...,China,7,47,232,6.714286,5,13,13,7,0
3,33719,73,7,207.860000,29.694286,0,0.000000,545,66,M,...,Brasil,7,54,73,7.714286,5,15,15,7,0
4,19742,12,6,232.889999,38.815000,3,0.500000,1030,65,M,...,United States,6,49,12,8.166667,5,14,14,6,1


---

**Next:** [02_propensity_modeling.ipynb](./02_propensity_modeling.ipynb) — EDA, model development, and business recommendations